[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/01-isochrones-getting-started.ipynb)

# Getting Started with Isochrones

An **isochrone** is a polygon showing the area reachable from a location within a given travel time. SocialMapper uses the [Valhalla](https://valhalla.github.io/valhalla/) routing engine (free, no API key required) to generate isochrones for driving, walking, and biking.

In this notebook you will learn how to:

1. Create your first isochrone from a city name
2. Inspect the GeoJSON result
3. Use coordinate tuples as input
4. Compare travel modes (drive / walk / bike)
5. Compare travel times
6. Visualize isochrones with matplotlib

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install socialmapper

from socialmapper import create_isochrone

## 1. Your First Isochrone

Pass a `"City, State"` string and SocialMapper geocodes it automatically.

In [ ]:
iso = create_isochrone("Nashville, TN", travel_time=15, travel_mode="drive")
print(type(iso))
print(iso["type"])

## 2. Inspect the GeoJSON Feature

`create_isochrone` returns a GeoJSON **Feature** dict with `type`, `geometry`, and `properties`.

In [ ]:
# Properties contain metadata about the isochrone
for key, value in iso["properties"].items():
    print(f"{key}: {value}")

In [ ]:
# The geometry is a GeoJSON Polygon
geom = iso["geometry"]
print(f"Geometry type: {geom['type']}")
print(f"Number of coordinate rings: {len(geom['coordinates'])}")
print(f"Vertices in outer ring: {len(geom['coordinates'][0])}")

## 3. Coordinate Tuple Input

You can also pass a `(latitude, longitude)` tuple directly.

In [ ]:
# Downtown Nashville coordinates
iso_coords = create_isochrone((36.1627, -86.7816), travel_time=10, travel_mode="drive")
print(f"Location: {iso_coords['properties']['location']}")
print(f"Area: {iso_coords['properties']['area_sq_km']:.1f} sq km")

## 4. Compare Travel Modes

SocialMapper supports three travel modes: `'drive'`, `'walk'`, and `'bike'`.

In [ ]:
modes = ["drive", "walk", "bike"]
mode_results = {}

for mode in modes:
    result = create_isochrone("Nashville, TN", travel_time=15, travel_mode=mode)
    area = result["properties"]["area_sq_km"]
    mode_results[mode] = result
    print(f"{mode:>5}: {area:>8.1f} sq km")

## 5. Compare Travel Times

See how the reachable area grows as travel time increases.

In [ ]:
travel_times = [5, 10, 15, 30]
time_results = {}

for minutes in travel_times:
    result = create_isochrone("Nashville, TN", travel_time=minutes, travel_mode="drive")
    area = result["properties"]["area_sq_km"]
    time_results[minutes] = result
    print(f"{minutes:>2} min drive: {area:>8.1f} sq km")

## 6. Quick Matplotlib Visualization

We can render any isochrone polygon with a few lines of matplotlib.

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import shape

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, mode in zip(axes, ["drive", "walk", "bike"]):
    polygon = shape(mode_results[mode]["geometry"])
    x, y = polygon.exterior.xy
    ax.fill(x, y, alpha=0.3)
    ax.plot(x, y, linewidth=1)
    area = mode_results[mode]["properties"]["area_sq_km"]
    ax.set_title(f"{mode} — {area:.1f} sq km")
    ax.set_aspect("equal")
    ax.tick_params(labelsize=7)

fig.suptitle("15-Minute Isochrones from Nashville, TN", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay multiple travel-time isochrones on one plot
fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#fee5d9", "#fcae91", "#fb6a4a", "#cb181d"]

# Plot largest first so smaller ones draw on top
for minutes, color in zip(reversed(travel_times), reversed(colors)):
    polygon = shape(time_results[minutes]["geometry"])
    x, y = polygon.exterior.xy
    ax.fill(x, y, alpha=0.5, color=color, label=f"{minutes} min")
    ax.plot(x, y, color=color, linewidth=0.8)

ax.set_title("Driving Isochrones from Nashville, TN")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Summary

| What you learned | API |
|---|---|
| Create an isochrone from a city name or coordinates | `create_isochrone(location, travel_time, travel_mode)` |
| Inspect GeoJSON Feature properties | `iso['properties']`, `iso['geometry']` |
| Compare drive / walk / bike reachability | `travel_mode='drive'` \| `'walk'` \| `'bike'` |

**Next notebook:** [02 — Census Block Groups](02-census-block-groups.ipynb)